In [ ]:
import json
import re
from urllib.parse import urljoin
import os
import requests


# =========================
# CONFIG
# =========================
SOURCE_BASE_URL = "https://dhis-pulse-dev.fhi360.org"
TARGET_BASE_URL = "https://dhis-pulse.fhi360.org"
PROGRAM_ID = ""

SOURCE_USERNAME = ""
SOURCE_PASSWORD = ""

TARGET_USERNAME = ""
TARGET_PASSWORD = ""

OUTPUT_DIR = r""
os.makedirs(OUTPUT_DIR, exist_ok=True)

PAYLOAD_PATH = os.path.join(OUTPUT_DIR, "dhis2_program_metadata_payload.json")
RESPONSE_PATH = os.path.join(OUTPUT_DIR, "dhis2_import_response.json")

# Change to False for Import, True for dry run
DRY_RUN = True

# CREATE_AND_UPDATE is default for metadata, but change if necessary
IMPORT_STRATEGY = "CREATE_AND_UPDATE"
ATOMIC_MODE = "ALL"



In [54]:

# =========================
# HELPERS
# =========================
def build_session(username: str, password: str) -> requests.Session:
    session = requests.Session()
    session.auth = (username, password)
    session.headers.update({
        "Accept": "application/json",
        "Content-Type": "application/json",
    })
    return session


def normalize_base_url(base_url: str) -> str:
    return base_url.rstrip("/") + "/"


def get_json(session: requests.Session, url: str) -> dict:
    resp = session.get(url)
    resp.raise_for_status()
    return resp.json()


def post_json(session: requests.Session, url: str, payload: dict) -> dict:
    resp = session.post(url, data=json.dumps(payload))

    # Try to parse the response body even when DHIS2 returns 409
    try:
        body = resp.json()
    except ValueError:
        body = {"raw_text": resp.text}

    if resp.status_code >= 400:
        print("\nHTTP ERROR:", resp.status_code)
        print(json.dumps(body, indent=2, ensure_ascii=False))
        resp.raise_for_status()

    return body

def chunk_list(items, chunk_size=50):
    for i in range(0, len(items), chunk_size):
        yield items[i:i + chunk_size]


def make_in_filter(ids):
    return f"id:in:[{','.join(ids)}]"


def extract_ids_from_list(objects, key="id"):
    ids = []
    for obj in objects:
        value = obj.get(key)
        if value:
            ids.append(value)
    return list(dict.fromkeys(ids))


def safe_get_collection(payload: dict, root_key: str):
    value = payload.get(root_key, [])
    return value if isinstance(value, list) else []


def plural_key_for_endpoint(endpoint_name: str) -> str:
    """
    Maps endpoint resource names to DHIS2 metadata payload keys.
    """
    mapping = {
        "programs": "programs",
        "programStages": "programStages",
        "programStageSections": "programStageSections",
        "dataElements": "dataElements",
        "optionSets": "optionSets",
        "options": "options",
        "dataEntryForms": "dataEntryForms",
        "programRuleVariables": "programRuleVariables",
        "programRules": "programRules",
        "programRuleActions": "programRuleActions",
    }
    return mapping[endpoint_name]


def build_metadata_url(base_url: str, path: str, query: str) -> str:
    return urljoin(normalize_base_url(base_url), f"api/{path}.json?{query}")


def fetch_collection(session: requests.Session, url: str, root_key: str):
    payload = get_json(session, url)
    return safe_get_collection(payload, root_key)


def fetch_by_ids(session: requests.Session, base_url: str, endpoint: str, ids, fields="*"):
    if not ids:
        return []
    root_key = plural_key_for_endpoint(endpoint)
    all_rows = []
    for batch in chunk_list(ids, 75):
        url = build_metadata_url(
            base_url,
            endpoint,
            f"filter={make_in_filter(batch)}&paging=false&fields={fields}"
        )
        rows = fetch_collection(session, url, root_key)
        all_rows.extend(rows)
    # dedupe by id
    deduped = {}
    for row in all_rows:
        row_id = row.get("id")
        if row_id:
            deduped[row_id] = row
    return list(deduped.values())


def parse_used_prv_names(program_rules):
    """
    Extract PRV references of the form #{PRV_NAME} from program rule conditions.
    """
    used = set()
    pattern = re.compile(r"#\{([^}]+)\}")

    for rule in program_rules:
        condition = rule.get("condition") or ""
        for match in pattern.findall(condition):
            used.add(match.strip())

    return used



In [55]:

# =========================
# URL GENERATORS
# =========================
def urls_for_program(base_url: str, program_id: str):
    base_url = normalize_base_url(base_url)

    urls = {}

    urls["program"] = build_metadata_url(
        base_url,
        "programs",
        f"filter=id:eq:{program_id}&paging=false&fields=*,!programIndicators,!organisationUnits"
    )

    urls["program_stage_lookup"] = build_metadata_url(
        base_url,
        "programs",
        f"filter=id:eq:{program_id}&paging=false&fields=programStages[id,programStageDataElements[dataElement[id]],dataEntryForm[id]]"
    )

    urls["program_rules"] = build_metadata_url(
        base_url,
        "programRules",
        f"filter=program.id:eq:{program_id}&paging=false&fields=*"
    )

    urls["program_rule_variables"] = build_metadata_url(
        base_url,
        "programRuleVariables",
        f"filter=program.id:eq:{program_id}&paging=false&fields=*"
    )

    return urls



In [56]:

# =========================
# METADATA COLLECTION
# =========================
def collect_program_metadata(source_session: requests.Session, base_url: str, program_id: str) -> dict:
    urls = urls_for_program(base_url, program_id)

    # 1) program
    programs = fetch_collection(source_session, urls["program"], "programs")
    if not programs:
        raise ValueError(f"No program found for ProgramID={program_id}")

    # 2) program stage lookup from the program object
    stage_lookup_payload = get_json(source_session, urls["program_stage_lookup"])
    lookup_programs = safe_get_collection(stage_lookup_payload, "programs")
    if not lookup_programs:
        raise ValueError("Program stage lookup returned no programs")

    lookup_program = lookup_programs[0]
    program_stages_min = lookup_program.get("programStages", [])

    program_stage_ids = []
    data_element_ids = []
    data_entry_form_ids = []

    for stage in program_stages_min:
        sid = stage.get("id")
        if sid:
            program_stage_ids.append(sid)

        data_entry_form = stage.get("dataEntryForm")
        if isinstance(data_entry_form, dict):
            dfid = data_entry_form.get("id")
            if dfid:
                data_entry_form_ids.append(dfid)

        for psde in stage.get("programStageDataElements", []) or []:
            de = psde.get("dataElement") or {}
            deid = de.get("id")
            if deid:
                data_element_ids.append(deid)

    program_stage_ids = list(dict.fromkeys(program_stage_ids))
    data_element_ids = list(dict.fromkeys(data_element_ids))
    data_entry_form_ids = list(dict.fromkeys(data_entry_form_ids))

    # 3) program stages
    program_stages = fetch_by_ids(
        source_session, base_url, "programStages", program_stage_ids, fields="*"
    )

    # 4) program stage sections
    # Common pattern: filter by ids from stages if programStageSections are nested there
    program_stage_section_ids = []
    for stage in program_stages:
        for section in stage.get("programStageSections", []) or []:
            sid = section.get("id")
            if sid:
                program_stage_section_ids.append(sid)
    program_stage_section_ids = list(dict.fromkeys(program_stage_section_ids))

    program_stage_sections = fetch_by_ids(
        source_session, base_url, "programStageSections", program_stage_section_ids, fields="*"
    )

    # 5) data elements
    data_elements = fetch_by_ids(
        source_session, base_url, "dataElements", data_element_ids, fields="*"
    )

    # 6) option sets from data elements
    option_set_ids = []
    for de in data_elements:
        option_set = de.get("optionSet")
        if isinstance(option_set, dict):
            osid = option_set.get("id")
            if osid:
                option_set_ids.append(osid)
    option_set_ids = list(dict.fromkeys(option_set_ids))

    option_sets = fetch_by_ids(
        source_session, base_url, "optionSets", option_set_ids, fields="*"
    )

    # 7) options from option sets
    option_ids = []
    for os in option_sets:
        for opt in os.get("options", []) or []:
            oid = opt.get("id")
            if oid:
                option_ids.append(oid)
    option_ids = list(dict.fromkeys(option_ids))

    options = fetch_by_ids(
        source_session, base_url, "options", option_ids, fields="*"
    )

    # 8) data entry forms
    data_entry_forms = fetch_by_ids(
        source_session, base_url, "dataEntryForms", data_entry_form_ids, fields="*"
    )

    # 9) program rules
    program_rules = fetch_collection(source_session, urls["program_rules"], "programRules")

    # 10) only used PRVs
    all_prvs = fetch_collection(source_session, urls["program_rule_variables"], "programRuleVariables")
    used_prv_names = parse_used_prv_names(program_rules)
    program_rule_variables = [
        prv for prv in all_prvs
        if prv.get("name") in used_prv_names
    ]

    # 11) program rule actions from program rule ids
    program_rule_ids = extract_ids_from_list(program_rules, key="id")
    program_rule_actions = []

    if program_rule_ids:
        for batch in chunk_list(program_rule_ids, 75):
            url = build_metadata_url(
                base_url,
                "programRuleActions",
                f"filter=programRule.id:in:[{','.join(batch)}]&paging=false&fields=*"
            )
            batch_rows = fetch_collection(source_session, url, "programRuleActions")
            program_rule_actions.extend(batch_rows)

        dedup = {}
        for row in program_rule_actions:
            rid = row.get("id")
            if rid:
                dedup[rid] = row
        program_rule_actions = list(dedup.values())

    # Assemble one metadata payload
    payload = {
        "programs": programs,
        "programStages": program_stages,
        "programStageSections": program_stage_sections,
        "dataElements": data_elements,
        "optionSets": option_sets,
        "options": options,
        "dataEntryForms": data_entry_forms,
        "programRuleVariables": program_rule_variables,
        "programRules": program_rules,
        "programRuleActions": program_rule_actions,
    }

    # remove empty collections
    payload = {k: v for k, v in payload.items() if v}

    return payload



In [57]:
# =========================
# IMPORT
# =========================
def import_metadata(
    target_session: requests.Session,
    target_base_url: str,
    payload: dict,
    dry_run: bool = True,
    import_strategy: str = "CREATE_AND_UPDATE",
    atomic_mode: str = "ALL",
):
    base = normalize_base_url(target_base_url)
    query = (
        f"importStrategy={import_strategy}"
        f"&atomicMode={atomic_mode}"
        f"&async=false"
    )

    if dry_run:
        query += "&dryRun=true"

    import_url = urljoin(base, f"api/metadata?{query}")
    return post_json(target_session, import_url, payload)


In [58]:
# =========================
# MAIN
# =========================
def main():
    source_session = build_session(SOURCE_USERNAME, SOURCE_PASSWORD)
    target_session = build_session(TARGET_USERNAME, TARGET_PASSWORD)

    payload = collect_program_metadata(
        source_session=source_session,
        base_url=SOURCE_BASE_URL,
        program_id=PROGRAM_ID,
    )

    print("Metadata object counts:")
    for key, value in payload.items():
        print(f"  {key}: {len(value)}")

    # Optional: save payload locally before import
    with open(PAYLOAD_PATH, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)

    result = import_metadata(
        target_session=target_session,
        target_base_url=TARGET_BASE_URL,
        payload=payload,
        dry_run=DRY_RUN,
        import_strategy=IMPORT_STRATEGY,
        atomic_mode=ATOMIC_MODE,
    )
    

    print("\nImport response:")
    print(json.dumps(result, indent=2, ensure_ascii=False))
    # Save response locally (for troubleshooting)
    with open(RESPONSE_PATH, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

if __name__ == "__main__":
    main()

Metadata object counts:
  programs: 1
  programStages: 1
  dataElements: 88
  optionSets: 10
  options: 95
  dataEntryForms: 1
  programRuleVariables: 3
  programRules: 3
  programRuleActions: 3

Import response:
{
  "httpStatus": "OK",
  "httpStatusCode": 200,
  "status": "OK",
  "response": {
    "responseType": "ImportReport",
    "stats": {
      "created": 0,
      "updated": 205,
      "deleted": 0,
      "ignored": 0,
      "total": 205
    },
    "typeReports": [
      {
        "klass": "org.hisp.dhis.programrule.ProgramRule",
        "stats": {
          "created": 0,
          "updated": 3,
          "deleted": 0,
          "ignored": 0,
          "total": 3
        },
        "objectReports": []
      },
      {
        "klass": "org.hisp.dhis.dataentryform.DataEntryForm",
        "stats": {
          "created": 0,
          "updated": 1,
          "deleted": 0,
          "ignored": 0,
          "total": 1
        },
        "objectReports": []
      },
      {
        "kla